# PWL近似(SOS2)— 適用前・原理・適用・効果

非凸な1変数関数(効率曲線・多峰なコスト曲線など)を最適化に組み込むと、そのままでは
非凸制約になり空間分枝が要る。実務では**区分線形(PWL)近似**で扱うが、区分の選択に
Big-M + バイナリを使うとバイナリ変数が区分数に比例して増える。

この notebook は次の型で PWL 近似を追う。

1. **課題(before)** — 非凸関数と、Big-M 近似がバイナリを大量に要すること
2. **原理(principle)** — 折れ点(breakpoints)と近似誤差、SOS2 が隣接性をバイナリ無しで表すこと
3. **適用(how)** — `mk.pwl_sos2` を1行当てる
4. **効果(before/after)** — SOS2 vs Big-M をモデル規模・求解で比較する

題材は `samples/physics_and_control_minlp/pwl_sos.py`。非凸・多峰の
`f(x)=sin(1.5x)+0.15(x−5)²` を PWL 近似して最小化する。
`build_model(method="sos2" | "bigm")` で SOS2 版と Big-M 版を切り替えられる。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探す。docs/samples/ があるため samples 有無では docs で止まる。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/physics_and_control_minlp"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import pwl_sos as pw

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — 非凸関数と Big-M のバイナリ

`f(x)` は多峰で非凸。凸ソルバの1回では最小に届かず、区分線形近似して整数計画に載せる。
Big-M 版は各区分に選択バイナリ `z_s` を置くため、**折れ点が K 個ならバイナリは K−1 個**。
下は SOS2 版と Big-M 版のバイナリ変数の数の比較。

In [2]:
def count_bin(m):
    return sum(1 for v in m.getVars() if v.vtype() == "BINARY")

for method in ("sos2", "bigm"):
    m = pw.build_model(method)
    print(f"{method:5s}: 変数 {m.getNVars():3d} / バイナリ {count_bin(m):2d} / 折れ点 {pw.N_BREAK}")

sos2 : 変数  23 / バイナリ  0 / 折れ点 21
bigm : 変数  43 / バイナリ 20 / 折れ点 21


## 2. 原理(principle) — 折れ点・近似誤差・SOS2

PWL 近似は折れ点 `(x_k, f(x_k))` を結ぶ折れ線。区分数を増やすほど近似は真の曲線に近づく。
下図に真の `f(x)` と、折れ点数を変えた PWL 近似を重ねる。

In [3]:
def pwl_eval(xq, xs, ys):
    return np.interp(xq, xs, ys)

xq = np.linspace(pw.X_LO, pw.X_HI, 400)
ftrue = np.array([pw.f(v) for v in xq])

fig = go.Figure(layout=base_layout(
    "非凸関数 f(x) の区分線形近似(折れ点数を変える)", "x", "f(x)", height=420))
fig.add_trace(go.Scatter(x=xq, y=ftrue, mode="lines",
    line=dict(color=C["ink"], width=2.5), name="真の f(x)"))
for nb_pt, col in zip([5, 9, 21], [C["warn"], C["s2"], C["s1"]]):
    xs = np.linspace(pw.X_LO, pw.X_HI, nb_pt)
    ys = np.array([pw.f(v) for v in xs])
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines+markers",
        line=dict(color=col, width=1.5, dash="dot"),
        marker=dict(color=col, size=6),
        name=f"PWL {nb_pt}点 (折れ点)"))
show(fig)

区分数が増えるほど誤差は下がる。最大近似誤差を折れ点数の関数として定量化する。

In [4]:
def max_err(nb_pt):
    xs = np.linspace(pw.X_LO, pw.X_HI, nb_pt)
    ys = np.array([pw.f(v) for v in xs])
    return float(np.max(np.abs(pwl_eval(xq, xs, ys) - ftrue)))

npts = list(range(3, 41, 2))
errs = [max_err(k) for k in npts]
bins_bigm = [k - 1 for k in npts]       # Big-M: バイナリ = 区分数 = K-1
bins_sos2 = [0 for _ in npts]           # SOS2: バイナリ 0

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=npts, y=errs, mode="lines+markers",
    line=dict(color=C["ink"], width=2.5), marker=dict(size=5),
    name="最大近似誤差"), secondary_y=False)
fig.add_trace(go.Scatter(x=npts, y=bins_bigm, mode="lines",
    line=dict(color=C["warn"], width=2, dash="dash"),
    name="Big-M のバイナリ数 (=K−1)"), secondary_y=True)
fig.add_trace(go.Scatter(x=npts, y=bins_sos2, mode="lines",
    line=dict(color=C["s1"], width=2),
    name="SOS2 のバイナリ数 (=0)"), secondary_y=True)
fig.update_layout(
    title=dict(text="精度 vs モデル規模: 折れ点を増やしても SOS2 はバイナリ0のまま",
               x=0.01, font=dict(color=C["ink"], size=15, family=FONT)),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=400,
    margin=dict(l=60, r=60, t=48, b=48), hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
fig.update_xaxes(title_text="折れ点数 K", gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_yaxes(title_text="最大近似誤差", gridcolor=C["grid"], linecolor=C["axis"],
                 zeroline=False, secondary_y=False)
fig.update_yaxes(title_text="バイナリ変数の数", zeroline=False, secondary_y=True)
show(fig)

**SOS2 が隣接性をバイナリ無しで表す。** PWL の重み `λ_k`(Σλ=1, λ≥0)は本来
「非ゼロなのは隣接する2点だけ」でなければならない。Big-M はこの隣接性を区分バイナリ `z_s` と
`λ_k ≤ z_{k−1}+z_k` で強制する。**SOS2 制約**(Special Ordered Set type 2: 非ゼロが高々2つ、
かつ隣接)は同じ性質を SCIP に直接渡す。SCIP は SOS2 専用の分枝規則を持つため、
補助バイナリを経由しない分だけモデルが軽い。上図のとおり折れ点を増やしても SOS2 の
バイナリは 0 のままである。

## 3. 適用(how) — `mk.pwl_sos2`

任意の1変数関数を、折れ点とその関数値を渡すだけで SOS2-PWL に置き換えられる。

```python
xs = [x0, x1, ..., xK]          # 折れ点(昇順)
ys = [f(x0), f(x1), ..., f(xK)] # 各折れ点での関数値
y = mk.pwl_sos2(m, x, xs, ys, name="fx")  # y ≈ f(x)。バイナリもBig-Mも使わない
```

下は最小の動作確認。`f(x)=x²` を3点で近似し、`x=1` での値を得る。

In [5]:
from pyscipopt import Model
m = Model(); m.hideOutput()
x = m.addVar(lb=0.0, ub=2.0, name="x")
y = mk.pwl_sos2(m, x, [0.0, 1.0, 2.0], [0.0, 1.0, 4.0], "sq")
m.addCons(x == 1.0)
m.setObjective(y, "minimize")
m.optimize()
print(f"x=1 での近似値 y = {m.getVal(y):.2f}  (f(1)=1.0)")

x=1 での近似値 y = 1.00  (f(1)=1.0)


## 4. 効果(before/after) — SOS2 vs Big-M

同じ折れ点で SOS2 版と Big-M 版を解き、最適値・モデル規模・求解を比較する。

In [6]:
df = mk.compare_variants(
    {"bigm": lambda: pw.build_model("bigm"),
     "sos2": lambda: pw.build_model("sos2")}, time_limit=10)

opt = {mth: pw.build_model(mth) for mth in ("bigm", "sos2")}
for mth, m in opt.items():
    m.hideOutput(); m.optimize()
bins = {mth: count_bin(pw.build_model(mth)) for mth in ("bigm", "sos2")}
print("最適値 bigm =", f"{opt['bigm'].getObjVal():.4f}",
      "/ sos2 =", f"{opt['sos2'].getObjVal():.4f}  (一致)")
df[["variant", "root_dual", "final_dual", "final_gap", "nodes", "time"]]

最適値 bigm = -0.5214 / sos2 = -0.5214  (一致)


,variant,root_dual,final_dual,final_gap,nodes,time
0,bigm,-0.521434,-0.521434,0.0,1,0.0
1,sos2,-0.521434,-0.521434,0.0,1,0.0


In [7]:
labels = ["Big-M", "SOS2"]
bar_colors = [C["warn"], C["s1"]]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.18,
    subplot_titles=("バイナリ変数の数 (少ないほど軽い)", "探索ノード数"))
fig.add_trace(go.Bar(x=labels, y=[bins["bigm"], bins["sos2"]], marker_color=bar_colors,
    text=[bins["bigm"], bins["sos2"]], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
nodes = [int(df.loc[df.variant == v, "nodes"].iloc[0]) for v in ("bigm", "sos2")]
fig.add_trace(go.Bar(x=labels, y=nodes, marker_color=bar_colors,
    text=nodes, textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="SOS2 vs Big-M の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

## まとめ

- SOS2 版と Big-M 版は**同じ最適値**に達する(等価な PWL 表現)。
- 違いは**モデル規模**。Big-M は折れ点数に比例してバイナリが増えるが、SOS2 は
  バイナリ 0 本で同じ隣接性を表す。SCIP が SOS2 専用の分枝規則を持つため、
  補助バイナリを経由しない分だけ探索が軽い。この小さな題材では両者とも即座に解けるが、
  PWL がモデルの多数の変数に埋め込まれると本数差がそのまま効いてくる。

### 効かないとき・注意

- **1変数関数専用**。多変数の非凸項(積など)には使えない → その場合は
  [整数×連続の厳密線形化](01_linearize_product.ipynb) や凸包再定式化を検討する。
- 折れ点を増やすほど近似精度は上がるが `λ` 変数も増える(精度とのトレードオフ)。
  上の「精度 vs モデル規模」図で必要精度に見合う折れ点数を選ぶ。

関連: [プレイブック 2. PWL近似(SOS2)](../../playbook/02-pwl-sos2.md) /
API [`mk.pwl_sos2`](../../api/transforms.md)